## variable selection looped over a range of rensonance-masses

### *BulkGrav reco-analysis*

### *–––– Set up ––––*

In [15]:
import numpy as np
import awkward as ak
import uproot
import matplotlib.pyplot as plt
import hist
import hist.dask as hda
import dask
import coffea.processor as processor
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
import vector
import json
import pandas as pd

NanoAODSchema.warn_missing_crossrefs = False

In [16]:
with open("samples.json", 'r') as f:
    fileset = json.load(f)

# pick out only the Bulk keys
bulk_samples = [name for name in fileset if name.startswith("BulkGravToWW")]

### *––– Helper Functions –––*

In [24]:
# -- Subjet Pair Extraction for Semi-Leptonic Events --
def extract_semi_subjet_pairs(fatjets, all_subjets, category_name="Semi-Leptonic"):
    pairs = []
    n_skipped = 0

    for event_fjs, event_sjs in zip(fatjets, all_subjets):
        if len(event_fjs) == 0:
            continue
        fj = event_fjs[0]
        try:
            idx1, idx2 = fj.subJetIdx1, fj.subJetIdx2
            if idx1 < 0 or idx2 < 0:
                raise ValueError("Invalid subjet indices")

            sj1, sj2 = event_sjs[idx1], event_sjs[idx2]
            vec1 = vector.obj(pt=sj1["pt"], eta=sj1["eta"], phi=sj1["phi"], mass=sj1["mass"])
            vec2 = vector.obj(pt=sj2["pt"], eta=sj2["eta"], phi=sj2["phi"], mass=sj2["mass"])
            pairs.append((vec1, vec2))
        except (KeyError, TypeError, ValueError):
            n_skipped += 1

    print(f"[{category_name}] Valid subjet pairs: {len(pairs)}, Skipped: {n_skipped}")
    return pairs

# -- Subjet Pair Extraction for Hadronic Events --
def extract_had_subjet_pairs(fatjets, category_name="Hadronic"):
    valid_pairs = []
    n_skipped = 0

    for subjets in fatjets.subjets:
        if len(subjets) < 2:
            n_skipped += 1
            continue
        try:
            sj1, sj2 = subjets[0], subjets[1]
            vec1 = vector.obj(pt=sj1["pt"], eta=sj1["eta"], phi=sj1["phi"], mass=sj1["mass"])
            vec2 = vector.obj(pt=sj2["pt"], eta=sj2["eta"], phi=sj2["phi"], mass=sj2["mass"])
            valid_pairs.append((vec1, vec2))
        except (KeyError, TypeError, ValueError):
            n_skipped += 1

    print(f"[{category_name}] Valid subjet pairs: {len(valid_pairs)}, Skipped: {n_skipped}")
    return valid_pairs

In [27]:
# -- Compute cos(θ*) for Given Subjet Pairs --
def compute_cos_theta_star(subjet_pairs, category_name=""):
    cos_theta_star = []
    n_success = 0
    n_skipped = 0

    for i, (vec1, vec2) in enumerate(subjet_pairs):
        # 1) Reconstruct the W four-vector in the lab frame
        w_lab = vec1 + vec2
        if w_lab.mass < 1e-3 or w_lab.E <= 1e-6:
            n_skipped += 1
            continue

        # 2) Build the boost to the W rest frame
        beta3 = -w_lab.to_beta3()
        if beta3.mag < 1e-6:
            n_skipped += 1
            continue
        
        # 3) RAPIDITY SORT: pick the subjet that’s more “backward” in η
        #    (this ensures the dot can go negative when it should)
        if vec1.rapidity < vec2.rapidity:
            q = vec1  
        else:
            q = vec2

        # 4) Boost that chosen subjet into the W rest frame
        q_rest = q.boost_beta3(beta3)
        if q_rest.to_beta3().mag < 1e-6:
            n_skipped += 1
            continue

        # 5) Compute signed cosθ* via vector’s unit-vector dot        
        w_hat  = w_lab.to_beta3().unit()
        q_hat  = q_rest.to_beta3().unit()
        cos_theta = q_hat.dot(w_hat)

        cos_theta_star.append(cos_theta)
        n_success += 1
    
    if n_success > 0:
            cos_array = np.array(cos_theta_star)
            print(f"[{category_name}] Computed cosθ* for {n_success} subjet pairs, Skipped: {n_skipped}")
            print(f"[{category_name}] range: min = {cos_array.min():.3f}, max = {cos_array.max():.3f}")
            print(f"[{category_name}] Histogram counts (neg, zero, pos): "
                  f"{np.sum(cos_array < 0)}, {np.sum(cos_array == 0)}, {np.sum(cos_array > 0)}")
    else:
        print(f"[{category_name}] All {n_skipped} entries were skipped; no valid cosθ*.")

    return np.array(cos_theta_star)

### *–––main–––*

In [ ]:
for sample_name in bulk_samples:

    # ––– STEP 0: LOAD EVENTS ––––
    events = NanoEventsFactory.from_root(
        fileset[sample_name]["files"],
        entry_stop=10000,
        metadata=fileset[sample_name]["metadata"],
        schemaclass=NanoAODSchema,
        delayed=False,
    ).events()

    # ––– STEP 1: TIGHT LEPTON SELECTION  –––
    muons = events.Muon
    electrons = events.Electron
    
    muons_tight = muons[
        (muons.pt > 35) & (muons.eta < 2.4) & (muons.tightId) &
        (((muons.pt < 10) & (muons.dxy < 0.01)) | ((muons.pt >= 10) & (muons.dxy < 0.02))) &
        (muons.dz < 0.05) & (muons.pfIsoId >= 4)
    ]
    
    electrons_tight = electrons[
        (electrons.pt > 30) & (electrons.eta < 2.4) & (((electrons.pt < 15) & 
        (electrons.dxy < 0.01)) | ((electrons.pt >= 15) & 
        (electrons.dxy < 0.02))) & (electrons.dxy < 0.05) & # dxy tighter (electrons.dz < 0.05) &
        (electrons.pfRelIso03_all < 0.15) & (electrons.lostHits <= 1) & (electrons.convVeto == True)
    ]
    
    tight_muon_count = ak.num(muons_tight)
    tight_electron_count = ak.num(electrons_tight)
    loose_muons = muons[(muons.isPFcand) & (muons.mediumId) & (muons.pfRelIso03_all < 0.4)]
    loose_muons = loose_muons[loose_muons.pt > 10]
    loose_electrons = electrons[electrons.cutBased >= 2]
    loose_electrons = loose_electrons[loose_electrons.pt > 10]
    loose_lepton_count = ak.num(loose_muons) + ak.num(loose_electrons)
    
    single_lepton_mask = ((tight_muon_count + tight_electron_count) == 1) & (loose_lepton_count == 1)
    events_selected = events[single_lepton_mask]
    
    print(f"{len(events_selected)} events {len(events)} passed the tight lepton selection and loose veto.")

    # Masks for exactly one tight lepton
    one_tight_electron_mask = (tight_electron_count == 1) & (tight_muon_count == 0) & (loose_lepton_count == 1)
    one_tight_muon_mask = (tight_muon_count == 1) & (tight_electron_count == 0) & (loose_lepton_count == 1)
    
    # Apply masks to original tight collections
    tight_electron_one = electrons_tight[one_tight_electron_mask]
    tight_muon_one = muons_tight[one_tight_muon_mask]

    # ––– STEP 2: AK8 CLEANING –––
    # FatJets cuts
    clean_fatJets = events.FatJet[(events.FatJet.pt > 150) & (events.FatJet.eta < 2.4) & (events.FatJet.msoftdrop > 0)]
    #Jets cuts
    clean_Jets = events.Jet[(events.Jet.pt > 30) & (events.Jet.eta < 4.6)]

    #Removing AK4(Jet) jets overlapping with AK8(FatJets) jets
    jets_fatjets = ak.cartesian({"x": clean_Jets, "y": clean_fatJets})
    jets_iso_f = ((jets_fatjets["x"].eta-jets_fatjets["y"].eta)**2+(jets_fatjets["x"].phi-jets_fatjets["y"].phi)**2>0.8**2)
    jets_fatjets = jets_fatjets[jets_iso_f]
    jets, fj = ak.unzip(jets_fatjets)

    # Require events with at least 1 clean AK8 FatJets
    AK8jets_candidates_mask = ak.num(clean_fatJets) >= 1
    Wjets_candidates = clean_fatJets[AK8jets_candidates_mask]
    leading_W_jet = Wjets_candidates[:, 0]
    leading_W_jet_pt = leading_W_jet.pt

    # Require events with at least  clean AK8 FatJets
    AK8jets_2_mask = ak.num(clean_fatJets) >= 2
    Wjets_2_candidates = clean_fatJets[AK8jets_2_mask]
    second_W_jet = Wjets_2_candidates[:, 1]
    second_W_jet_pt = second_W_jet.pt

    # ––– STEP 3: EVENT CATEGORIZATION –––
    #tight lepton masks
    n_tight_muons = ak.num(muons_tight)
    n_tight_electrons = ak.num(electrons_tight)
    
    # event masks
    semi_leptonic_mask = (
        ((n_tight_muons == 1) & (n_tight_electrons == 0)) |
        ((n_tight_muons == 0) & (n_tight_electrons == 1))) & (ak.num(clean_fatJets) >= 1)
    
    fully_hadronic_mask = (
        (n_tight_muons == 0) & (n_tight_electrons == 0) &
        (ak.num(clean_fatJets) >= 2))

    # semi-leptonic W decay: 
    semi_leptonic_fatjets = clean_fatJets[semi_leptonic_mask]
    # fully hadronic W decay: 
    fully_hadronic_fatjets = clean_fatJets[fully_hadronic_mask]

    # For plotting (flatten to leading/subleading jets)
    semi_leptonic_leading_jet = semi_leptonic_fatjets[:, 0]
    fully_hadronic_leading_jet = fully_hadronic_fatjets[:, 0]
    fully_hadronic_second_jet = fully_hadronic_fatjets[:, 1]

    # STEP 4: COMPUTE cosθ* FOR EACH CATEGORY –––
    semi_pairs = extract_semi_subjet_pairs(semi_leptonic_fatjets, events.SubJet[semi_leptonic_mask], "Semi-Leptonic")
    had1_pairs = extract_had_subjet_pairs(fully_hadronic_leading_jet, "Hadronic Lead")
    had2_pairs = extract_had_subjet_pairs(fully_hadronic_second_jet, "Hadronic Sublead")

    cos_theta_semi = compute_cos_theta_star(semi_pairs, "Semi-Leptonic")
    cos_theta_had1 = compute_cos_theta_star(had1_pairs, "Hadronic Lead")
    cos_theta_had2 = compute_cos_theta_star(had2_pairs, "Hadronic Sublead")

    # — STEP 5: BUILD per‑channel DataFrames and TAG longitudinal (bulk) samples —
    # semi‑leptonic AK8 candidate (one per event)
    df_semi = pd.DataFrame({
        "pt":    ak.to_numpy(semi_leptonic_leading_jet.pt),
        "eta":   ak.to_numpy(semi_leptonic_leading_jet.eta),
        "msoft": ak.to_numpy(semi_leptonic_leading_jet.msoftdrop),
        "cos":   cos_theta_semi,
    })
    df_semi["channel"] = "semi"
    df_semi["label"]   = 1      # 1 → longitudinal (Bulk)

    # fully‐hadronic AK8 candidates
    df_had = pd.DataFrame({
        "pt1":    ak.to_numpy(fully_hadronic_leading_jet.pt),
        "eta1":   ak.to_numpy(fully_hadronic_leading_jet.eta),
        "msoft1": ak.to_numpy(fully_hadronic_leading_jet.msoftdrop),
        "cos1":   cos_theta_had1,
    })
    df_had["channel"] = "hadronic"
    df_had["label"]   = 1

    # — STEP 6: CONCATENATE and SAVE two .h5 files for both decay channels per resonance mass—
    df_out = pd.concat([df_semi, df_had], ignore_index=True)
    outname = f"{sample_name}_features.h5"
    df_out.to_hdf(outname, key="df", mode="w")
    print(f"Wrote {outname}: {len(df_out)} entries")